# Retriever experiments

This notebook evaluates the candidate-generation stage of Shiki RecSys.

Goals:

- compare Popularity, Explicit SVD, Implicit ALS, and TF-IDF retrievers;
- measure Candidate Recall@K and Catalog Coverage@K;
- examine overlap and complementarity between retrievers;
- estimate the upper bound provided by their combined candidate pool;
- use validation data only for model selection and preserve the test split for final evaluation.

In [1]:
import os
from pathlib import Path

import pandas as pd

from shiki_recsys.config.settings import get_settings
from shiki_recsys.config.training import load_training_config
from shiki_recsys.database.session import (
    create_database_engine,
    create_session_factory,
)
from shiki_recsys.evaluation.retrievers import (
    evaluate_retriever_run,
    generate_retriever_evaluation_run,
)
from shiki_recsys.preprocessing.splitting import chronological_split
from shiki_recsys.training.data import load_training_data
from shiki_recsys.training.retrievers import fit_retrievers

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

PROJECT_ROOT

PosixPath('/home/nika/shiki_recsys')

## Data and evaluation protocol

Retrievers are trained on the chronological training split and evaluated against positive interactions from the validation period.

The held-out test split is not used during retriever selection or hyperparameter tuning.

In [2]:
loaded_config = load_training_config(PROJECT_ROOT / "configs" / "training.yaml")

config = loaded_config.model_copy(
    update={
        "random_seed": 42,
        "dataset": loaded_config.dataset.model_copy(
            update={
                "split": loaded_config.dataset.split.model_copy(
                    update={
                        "validation_fraction": 0.1,
                        "test_fraction": 0.1,
                        "min_interactions_per_user": 10,
                    }
                )
            }
        ),
        "target": loaded_config.target.model_copy(
            update={
                "positive_rating_threshold": 8,
            }
        ),
        "retrievers": loaded_config.retrievers.model_copy(
            update={
                "explicit_svd": (
                    loaded_config.retrievers.explicit_svd.model_copy(
                        update={
                            "min_item_explicit_ratings": 5,
                            "n_factors": 64,
                            "n_epochs": 15,
                            "biased": True,
                            "learning_rate": 0.015,
                            "regularization": 0.10,
                            "init_mean": 0.0,
                            "init_std_dev": 0.1,
                        }
                    )
                ),
                "implicit_als": (
                    loaded_config.retrievers.implicit_als.model_copy(
                        update={
                            "signal_confidences": (
                                loaded_config.retrievers.implicit_als.signal_confidences.model_copy(
                                    update={
                                        "rating_8_10": 2.0,
                                        "watching": 1.0,
                                        "rewatching": 1.0,
                                        "completed": 0.5,
                                        "planned": 0.5,
                                        "on_hold": 0.0,
                                        "rating_4_5": -1.0,
                                        "rating_1_3": -2.0,
                                    }
                                )
                            ),
                            "factors": 96,
                            "regularization": 0.1,
                            "alpha": 1.0,
                            "iterations": 40,
                        }
                    )
                ),
                "content_tfidf": (
                    loaded_config.retrievers.content_tfidf.model_copy(
                        update={
                            "max_positive_items": 50,
                        }
                    )
                ),
            }
        ),
        "candidate_generation": (
            loaded_config.candidate_generation.model_copy(
                update={
                    "retrieval_k": 100,
                }
            )
        ),
    }
)
settings = get_settings()

engine = create_database_engine(settings)
session_factory = create_session_factory(engine)

try:
    with session_factory() as session:
        data = load_training_data(session)
finally:
    engine.dispose()

interactions = data.interactions
catalog = data.catalog

split_config = config.dataset.split

split = chronological_split(
    interactions,
    validation_fraction=split_config.validation_fraction,
    test_fraction=split_config.test_fraction,
    min_interactions_per_user=split_config.min_interactions_per_user,
)

pd.Series(
    {
        "train_interactions": len(split.train),
        "validation_interactions": len(split.validation),
        "test_interactions": len(split.test),
        "catalog_items": len(catalog),
        "retrieval_k": config.candidate_generation.retrieval_k,
        "positive_rating_threshold": (config.target.positive_rating_threshold),
    },
    name="value",
)

train_interactions           875697
validation_interactions      108915
test_interactions            108915
catalog_items                 23802
retrieval_k                     100
positive_rating_threshold         8
Name: value, dtype: int64

## Baseline retrievers

The first experiment evaluates the current retriever configuration without modifying any hyperparameters. This establishes the baseline against which later experiments are compared.

In [3]:
(
    popularity,
    explicit_svd,
    implicit_als,
    content_tfidf,
) = fit_retrievers(
    interactions=split.train,
    catalog=catalog,
    config=config,
)

In [4]:
candidate_count = config.candidate_generation.retrieval_k

baseline_run = generate_retriever_evaluation_run(
    history_interactions=split.train,
    target_interactions=split.validation,
    positive_rating_threshold=(config.target.positive_rating_threshold),
    candidate_count=candidate_count,
    popularity=popularity,
    explicit_svd=explicit_svd,
    implicit_als=implicit_als,
    content_tfidf=content_tfidf,
)

In [5]:
baseline_evaluation = evaluate_retriever_run(
    baseline_run,
    candidate_count=candidate_count,
    catalog_size=len(catalog),
)

baseline_evaluation.individual_metrics

,source,recall_at_k,catalog_coverage_at_k,evaluated_users
0,popularity,0.274939,0.046929,952
1,explicit_svd,0.211365,0.078229,952
2,implicit_als,0.331908,0.158936,952
3,content_tfidf,0.103757,0.189396,952


### Baseline observations

Implicit ALS provides the strongest individual candidate retrieval, achieving the highest Recall@100 while maintaining substantially broader catalog coverage than the popularity and SVD baselines.

Popularity is a strong non-personalized baseline, but its recommendations cover a relatively narrow portion of the catalog.

Explicit SVD provides lower recall than both ALS and popularity under the current configuration.

TF-IDF has the lowest individual recall but the highest catalog coverage. Its value therefore depends on whether it retrieves relevant items that the collaborative retrievers miss, which is examined through candidate-pool complementarity below.

## Retriever complementarity

A hybrid candidate generator is useful only if its retrievers contribute different relevant items. Individual Recall@K alone cannot determine whether a weaker retriever is redundant.

The following analysis measures:

- recall of the union of all retriever candidate sets;
- recall lost when each retriever is removed;
- relevant items found uniquely by each retriever;
- pairwise candidate overlap.

In [6]:
best_individual_recall = baseline_evaluation.individual_metrics["recall_at_k"].max()

pd.Series(
    {
        "best_individual_recall": best_individual_recall,
        "union_recall": baseline_evaluation.union_recall,
        "union_lift": (baseline_evaluation.union_recall - best_individual_recall),
    },
    name="value",
)

best_individual_recall    0.331908
union_recall              0.542878
union_lift                0.210970
Name: value, dtype: float64

In [7]:
contribution_table = pd.DataFrame(
    [
        {
            "source": contribution.source.value,
            "full_union_recall": contribution.full_recall,
            "recall_without_source": (contribution.recall_without_source),
            "marginal_recall": contribution.marginal_recall,
            "unique_relevant_hits": (contribution.unique_relevant_hits),
        }
        for contribution in baseline_evaluation.contributions
    ]
)

contribution_table

,source,full_union_recall,recall_without_source,marginal_recall,unique_relevant_hits
0,popularity,0.542878,0.483056,0.059821,1404
1,explicit_svd,0.542878,0.477386,0.065492,1484
2,implicit_als,0.542878,0.441179,0.101698,1755
3,content_tfidf,0.542878,0.502098,0.040780,913


In [8]:
baseline_evaluation.jaccard_matrix

,popularity,explicit_svd,implicit_als,content_tfidf
popularity,1.000000,0.115459,0.201694,0.021503
explicit_svd,0.115459,1.000000,0.080259,0.014909
implicit_als,0.201694,0.080259,1.000000,0.033009
content_tfidf,0.021503,0.014909,0.033009,1.000000


### Complementarity observations

The combined candidate pool substantially outperforms every individual retriever. Its recall reaches 0.543 compared with 0.332 for the strongest standalone model, Implicit ALS.

Implicit ALS provides the largest marginal contribution and the largest number of uniquely retrieved relevant items, confirming its role as the strongest personalized candidate generator.

Explicit SVD is weaker than popularity when evaluated independently, but contributes more marginal recall to the ensemble. This shows that standalone retrieval quality does not fully describe a retriever's usefulness in a hybrid system.

TF-IDF has the weakest individual recall, but its candidate overlap with the other retrievers is extremely low and it retrieves 913 relevant user-item pairs that no other retriever finds. Removing it reduces union recall from 0.543 to 0.502.

These results support retaining all four retrievers in the candidate-generation stage. Their value comes not only from individual retrieval quality but also from complementary coverage of relevant items.

The union recall is a candidate-availability metric rather than a ranking metric. With up to 100 candidates contributed by each retriever, the union may contain substantially more than 100 items per user. It therefore represents the relevant-item coverage available to the downstream ranker, not Recall@100 of the final recommender.